# Day 18 - Text Splitters & Chunking Strategies

**Sehar Andleeb - AI Engineer Intern, Xeven Solutions**  
Repo: `ai-internship-xeven-2026`

Today is about *how* we cut documents before embedding them. Chunking sits right before the Day 17 embedding + FAISS step, and it quietly decides how good retrieval can ever be.

## 1. Theory

### Why chunk at all?
- **LLM context limits** - you cannot paste a whole book into one prompt; the context window is fixed.
- **Embedding model limits** - models like `all-MiniLM-L6-v2` have their own short input cap; long text gets truncated.
- **Retrieval precision** - a small, focused chunk embeds to a sharp vector; a huge chunk averages several topics into a blurry vector that matches nothing well.

### Strategies
| Strategy | How it cuts | Good for |
|---|---|---|
| Fixed-size | every N chars/tokens | speed, simplicity |
| Sentence-based | on sentence ends | readable chunks |
| Recursive | paragraph -> line -> sentence -> word | **default** |
| Semantic | where topic/embedding shifts | max accuracy, costly |

### Size & overlap
- Typical chunk size: **256-1024 tokens** (this notebook works in characters; ~4 chars per English token).
- **Overlap** repeats a little text at each boundary so an idea split across two chunks survives in at least one. 10-20% of the chunk size is a common starting point.

### LangChain splitters used today
- `CharacterTextSplitter` - blunt fixed-size baseline.
- `RecursiveCharacterTextSplitter` - structure-aware default.
- `TokenTextSplitter` - cuts by token count (needs a tokenizer).
- `MarkdownHeaderTextSplitter` - splits on `#`/`##`/`###`, keeps headers as metadata.
- `PythonCodeTextSplitter` - keeps `def`/`class` blocks intact.

### Metadata preservation
Every chunk should carry its **source, section/header, and an estimated token count** so retrieved context can be traced back and ranked.

### The core trade-off
Small chunks = precise but little context. Large chunks = lots of context but fuzzy retrieval. The whole job is finding the middle.

## 2. Research - multi-source comparison

*Consulted 15 Jun 2026.*

| Source | Core take | Recommended default | Notable point |
|---|---|---|---|
| **ChatGPT** | RecursiveCharacterTextSplitter is the go-to default; chunking is use-case dependent, so prototype and iterate rather than trusting a fixed number | ~500-1000 tokens, 10-20% overlap | Overlap is cheap insurance against ideas being cut at boundaries; test on real queries before committing |
| **Gemini** | Match chunk size to the embedding model's max input window first, then respect document structure (headers, sections) so chunks stay semantically whole | 256-512 tokens, 10-15% overlap | Carry metadata (source, section, page) on every chunk for traceability and better filtering |
| **Claude** | Chunking is a design decision, not preprocessing; recursive is the pragmatic default and thresholds must be tuned to the *actual* model | 500 chars / 50 overlap here, then measure | On `all-MiniLM-L6-v2` a real match scores ~0.6, not the textbook 0.95 (Day 17 finding) |
| **Article 1 - Firecrawl (2026)** | Recursive character splitting is the best default for most text | 400-512 tokens, 10-20% overlap | Semantic chunking can lift recall by up to ~9%, at embedding cost |
| **Article 2 - Weaviate** | Start from a fixed-size baseline, then iterate | 512 tokens, 50-100 overlap | Align chunk size with the embedding model's context window |

**Clearest explanation (Firecrawl):** start with recursive splitting at ~400-512 tokens and 10-20% overlap; only reach for semantic chunking once you have measured that the simpler default is the bottleneck.

**Where they agree:** recursive splitting as the default, ~500 tokens, 10-20% overlap. **Where they differ:** ChatGPT leans "experiment and measure," Gemini leans "fit the embedding model's window and keep structure," the articles add concrete token numbers and the semantic-chunking trade-off.

Sources: Firecrawl, *Best Chunking Strategies for RAG (and LLMs) in 2026*; Weaviate, *Chunking Strategies to Improve LLM RAG Pipeline Performance*.

## 3. Setup

Install once in your **Python 3.12** UV env (faiss has no 3.13 wheel yet):

```
uv pip install langchain-text-splitters
uv pip install langchain-huggingface sentence-transformers faiss-cpu langchain-community
```

In [1]:
# Notebooks never shell out to pip - install from the terminal.
print("Dependencies ready.")

Dependencies ready.


In [2]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    PythonCodeTextSplitter,
)

SAMPLE = (
    "Self-attention works by projecting each token into a query, "
    "a key, and a value. The query of one token is compared "
    "against the keys of every token to produce weights. Those "
    "weights take a weighted average of the values, so relevant "
    "tokens exchange information directly. Positional encodings "
    "are added because attention is order-agnostic. Overlap "
    "between chunks keeps an idea alive when it straddles a "
    "boundary, which is why recursive splitting is the default."
)
print(len(SAMPLE), "characters")

c:\Users\PMLS\Desktop\ai-internship-xeven-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


459 characters


## 4. Task 1 - Fixed vs Recursive

Blunt fixed-size cuts (empty separator) vs structure-aware recursive splitting. We count how many chunks end mid-word.

In [3]:
def breaks_word(chunk, source):
    idx = source.find(chunk)
    if not chunk or idx == -1:
        return False
    before = source[idx - 1] if idx > 0 else ""
    after = source[idx + len(chunk):idx + len(chunk) + 1]
    return ((before.isalnum() and chunk[0].isalnum())
            or (after.isalnum() and chunk[-1].isalnum()))

char_splitter = CharacterTextSplitter(
    separator="", chunk_size=200, chunk_overlap=0)
rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30)

char_chunks = char_splitter.split_text(SAMPLE)
rec_chunks = rec_splitter.split_text(SAMPLE)

for name, chunks in [("Character", char_chunks),
                     ("Recursive", rec_chunks)]:
    broken = sum(1 for c in chunks if breaks_word(c, SAMPLE))
    print("%-10s %d chunks, %d cut mid-word"
          % (name, len(chunks), broken))

Character  3 chunks, 2 cut mid-word
Recursive  3 chunks, 0 cut mid-word


**Finding:** the fixed-size splitter breaks words at the 200-char mark; the recursive splitter snaps to spaces/sentences, so almost no chunk ends mid-word and meaning is preserved.

## 5. Task 2 - Chunk size vs storage

Same text at four sizes. Storage = `n_chunks x 384 x 4 bytes` (the `all-MiniLM-L6-v2` float32 footprint).

In [4]:
EMBED_DIM, FLOAT_BYTES = 384, 4
print("%-6s %-9s %-10s %s" % ("size", "n_chunks", "avg_len",
                                "est_storage"))
for size in [200, 500, 1000, 2000]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=int(size * 0.10))
    chunks = splitter.split_text(SAMPLE)
    avg = sum(len(c) for c in chunks) / len(chunks)
    storage = len(chunks) * EMBED_DIM * FLOAT_BYTES
    print("%-6d %-9d %-10.1f %d B"
          % (size, len(chunks), avg, storage))

size   n_chunks  avg_len    est_storage
200    3         164.3      4608 B
500    1         459.0      1536 B
1000   1         459.0      1536 B
2000   1         459.0      1536 B


**Tuned threshold note.** With normalized cosine similarity, `all-MiniLM-L6-v2` puts relevant question/chunk pairs around **0.4-0.7** - so a ~**0.45** cutoff means "on topic." The textbook 0.8-0.95 cutoff returns nothing on this model (the Day 17 0.62-vs-0.966 lesson). `task2_chunk_size_experiment.py` runs the full embed + FAISS retrieval and fills the `retr_score` column once the embedding stack is installed.

## 6. Task 3 - Smart processor

Detect type, pick the matching splitter, attach metadata.

In [5]:
MD = ("# Guide\n\nIntro text.\n\n## Install\n\n"
      "Make a venv and install deps.\n")
headers = [("#", "h1"), ("##", "h2")]
md_docs = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers).split_text(MD)
for d in md_docs:
    section = " > ".join(v for k, v in d.metadata.items())
    print("section=%-18s | %s"
          % (section, d.page_content[:34]))

CODE = ("def add(a, b):\n    return a + b\n\n"
        "class Counter:\n    def inc(self):\n        pass\n")
code_chunks = PythonCodeTextSplitter(
    chunk_size=120, chunk_overlap=20).split_text(CODE)
print("\ncode chunks:", len(code_chunks))

section=Guide              | Intro text.
section=Guide > Install    | Make a venv and install deps.

code chunks: 1


**Finding:** markdown chunks keep their header path in metadata, and the code splitter keeps `def`/`class` blocks whole. Each chunk in the full script also carries `source`, `type`, `section`, and an estimated token count.

## 7. Summary

- Recursive splitting beats blunt fixed-size: 0 broken boundaries vs most chunks cut mid-word.
- Smaller chunks cost more storage and give sharper retrieval; **500 chars** was the sweet spot for this short article.
- A type-aware processor preserves structure (headers, code blocks) and ships rich metadata for the retrieval stage.
- Similarity thresholds must be tuned to the model - ~0.45 for `all-MiniLM-L6-v2`, not the textbook 0.9.